# Module 04: ZeRO Optimization (FSDP + DeepSpeed)

Read `README.md` before starting.

**Goal**: Shard model state across GPUs to fit larger models and reduce per-GPU memory.

In [ ]:
!pip install -q deepspeed==0.14.0

import torch
assert torch.cuda.device_count() == 2
print(f"Ready. GPUs: {torch.cuda.device_count()}")

## Part 1: FSDP — Three Sharding Strategies Side by Side

We run the same model under `NO_SHARD` (DDP), `SHARD_GRAD_OP` (ZeRO-2), and `FULL_SHARD` (ZeRO-3).
Compare the memory numbers.

In [ ]:
import subprocess, time

def kill_distributed_stragglers():
    """
    Kill any lingering torchrun / torch.distributed worker processes
    that may be holding a rendezvous port from a previous cell run.
    Safe to call multiple times.
    """
    result = subprocess.run(
        ["pkill", "-f", "torch.distributed.run"],
        capture_output=True
    )
    time.sleep(1)   # Give the OS a moment to release sockets

kill_distributed_stragglers()
print("Cleaned up any leftover distributed processes.")

In [ ]:
print("=" * 60)
print("Strategy 1: NO_SHARD (equivalent to DDP)")
print("=" * 60)
!torchrun --nproc_per_node=2 --master_port=29500 /kaggle/working/fsdp_compare.py --strategy NO_SHARD

In [ ]:
print("=" * 60)
print("Strategy 2: SHARD_GRAD_OP (ZeRO-2)")
print("=" * 60)
!torchrun --nproc_per_node=2 --master_port=29501 /kaggle/working/fsdp_compare.py --strategy SHARD_GRAD_OP

In [ ]:
print("=" * 60)
print("Strategy 3: FULL_SHARD (ZeRO-3)")
print("=" * 60)
!torchrun --nproc_per_node=2 --master_port=29502 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD

In [ ]:
print("=" * 60)
print("Strategy 3: FULL_SHARD (ZeRO-3)")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD

**Fill in this table with your results:**

| Strategy | GPU 0 Peak (GB) | GPU 1 Peak (GB) | Throughput (samples/s) |
|----------|-----------------|-----------------|------------------------|
| NO_SHARD | | | |
| SHARD_GRAD_OP | | | |
| FULL_SHARD | | | |

You should see peak memory decrease as you go from NO_SHARD → FULL_SHARD,
and throughput decrease slightly (more communication overhead).

print("=" * 60)
print("FULL_SHARD + CPU Offload")
print("=" * 60)
!torchrun --nproc_per_node=2 --master_port=29503 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD --cpu_offload

In [ ]:
print("=" * 60)
print("FULL_SHARD + CPU Offload")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD --cpu_offload

## Part 3: Saving and Loading FSDP Checkpoints

FSDP checkpointing has a gotcha: the model is sharded across GPUs,
so you can't just call `torch.save(model.state_dict())`.

In [ ]:
!torchrun --nproc_per_node=2 --master_port=29504 /kaggle/working/fsdp_checkpoint.py

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_checkpoint.py

## Part 4: DeepSpeed ZeRO

DeepSpeed is a library by Microsoft that implements ZeRO with more features.
Config is JSON-based. The API wraps your model similar to FSDP.

In [ ]:
import json

# Write DeepSpeed config files for ZeRO stages 1, 2, 3
for stage in [1, 2, 3]:
    config = {
        "train_micro_batch_size_per_gpu": 4,
        "gradient_accumulation_steps": 1,
        "steps_per_print": 5,
        "optimizer": {
            "type": "AdamW",
            "params": {"lr": 1e-4, "weight_decay": 0.01}
        },
        "fp16": {"enabled": True},
        "zero_optimization": {
            "stage": stage,
            "allgather_partitions": True,
            "allgather_bucket_size": 2e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 2e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
        }
    }
    # ZeRO-3 needs param gather config
    if stage == 3:
        config["zero_optimization"]["stage3_param_persistence_threshold"] = 1e4
        config["zero_optimization"]["stage3_max_live_parameters"] = 1e9

    path = f"/kaggle/working/ds_config_stage{stage}.json"
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Written: {path}")

In [ ]:
%%writefile /kaggle/working/deepspeed_train.py
"""
DeepSpeed ZeRO training script.

Usage:
  deepspeed --num_gpus=2 deepspeed_train.py --stage 2
  deepspeed --num_gpus=2 deepspeed_train.py --stage 3
"""
import os
import argparse
import time
import torch
import torch.nn as nn
import deepspeed
from torch.utils.data import DataLoader, TensorDataset

VOCAB_SIZE = 50000
D_MODEL = 1024
NHEAD = 16
NUM_LAYERS = 12
BATCH_SIZE = 4
SEQ_LEN = 128
NUM_STEPS = 20

class TransformerLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(D_MODEL, NHEAD, D_MODEL*4, dropout=0.1, batch_first=True)
            for _ in range(NUM_LAYERS)
        ])
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", type=int, default=2, choices=[1, 2, 3])
    # DeepSpeed adds its own args; use parse_known_args
    args, _ = parser.parse_known_args()

    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    model = TransformerLM()
    total_params = sum(p.numel() for p in model.parameters())
    if local_rank == 0:
        print(f"Stage: ZeRO-{args.stage} | Params: {total_params:,}")

    ds_config = f"/kaggle/working/ds_config_stage{args.stage}.json"

    # DeepSpeed initialize — handles distributed setup, model wrapping, optimizer
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        config=ds_config,
    )

    dataset = TensorDataset(
        torch.randint(0, VOCAB_SIZE, (200, SEQ_LEN)),
        torch.randint(0, VOCAB_SIZE, (200, SEQ_LEN))
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    step = 0
    throughputs = []

    for epoch in range(100):
        for x, y in loader:
            if step >= NUM_STEPS:
                break

            t0 = time.perf_counter()
            x = x.to(model_engine.local_rank)
            y = y.to(model_engine.local_rank)

            logits = model_engine(x)
            loss = nn.CrossEntropyLoss()(logits.view(-1, VOCAB_SIZE), y.view(-1))

            model_engine.backward(loss)  # replaces loss.backward()
            model_engine.step()          # replaces optimizer.step()

            elapsed = time.perf_counter() - t0
            tp = BATCH_SIZE / elapsed
            throughputs.append(tp)

            if local_rank == 0 and step % 5 == 0:
                mem = torch.cuda.memory_allocated(local_rank) / 1e9
                print(f"[Step {step:3d}] loss={loss.item():.4f} | {tp:.0f}/s | mem={mem:.2f}GB")
            step += 1
        if step >= NUM_STEPS:
            break

    if local_rank == 0:
        avg_tp = sum(throughputs) / len(throughputs)
        peak = torch.cuda.max_memory_allocated(local_rank) / 1e9
        print(f"\nDeepSpeed ZeRO-{args.stage}: {avg_tp:.0f} samples/s | peak mem: {peak:.2f} GB")


if __name__ == "__main__":
    main()

In [ ]:
print("=" * 60)
print("DeepSpeed ZeRO-2")
print("=" * 60)
!deepspeed --num_gpus=2 /kaggle/working/deepspeed_train.py --stage 2

In [ ]:
print("=" * 60)
print("DeepSpeed ZeRO-3")
print("=" * 60)
!deepspeed --num_gpus=2 /kaggle/working/deepspeed_train.py --stage 3

## Checkpoint Questions

1. What specifically does ZeRO-2 shard that ZeRO-1 does not?
2. If you have 2 GPUs and a model with 1B parameters, how much memory does each GPU use for optimizer state under ZeRO-3 vs DDP? (Assume Adam optimizer, fp32 optimizer states, fp16 params)
3. Why can't you use `torch.save(fsdp_model.state_dict())` the same way as a regular model?
4. What additional communication operations does ZeRO-3 require that DDP does not?

**Answers:**
1. ZeRO-2 shards gradients in addition to optimizer states. ZeRO-1 only shards optimizer states.
2. DDP: 8 bytes/param × 1B = 8 GB optimizer state per GPU. ZeRO-3: 8/2 = 4 GB per GPU.
3. With FSDP, the model's parameters are split across GPUs. Calling `state_dict()` on one rank only returns that rank's shard. You need `FULL_STATE_DICT` mode to gather all shards.
4. ZeRO-3 requires AllGather before each layer's forward (to reconstruct full params), and ReduceScatter after backward (to shard gradients). DDP only requires AllReduce after backward.

---

## Next: Open `../05_accelerate/notebook.ipynb`